# StyleTTS2 Training Notebook

This notebook aims to help with training [StyleTTS2](https://github.com/yl4579/StyleTTS2) models. StyleTTS2 is an advanced open-source TTS system that can allows us to train human-like sounding models with emotions and context awareness.

StyleTTS2 requires a fair amount of GPU power to train models quickly and efficiently. While this Notebook can be ran on a Google Colab's T4, I [wouldn't recommend](https://github.com/yl4579/StyleTTS2/discussions/128#discussioncomment-7767318) doing any real work apart from trying StyleTTS2 out that way.

# Prerequisities

**Raw WAV files preparation**

- create audio files that are **longer than 1 second** and ideally **shorter than 12 - 20 seconds** in length along with **Audacity-compatible labels** ( you can use [My Audio Preparation Notebook](https://colab.research.google.com/drive/1ZK-2lAV2DokrN92sYPJRl47XyuuluCJe?usp=sharing) and then [Audacity](https://www.audacityteam.org/) to re-align the labels, if you have a single long recording that you need to transcribe )
- name your files, so they begin with the **same prefix** and always **have 5 digits** after it (such as B1-, e.g. *B1-00001.wav*)
- **zip-up** your **WAV files and labels.txt file** into a single ZIP file. All files need to be in the root folder of that ZIP file, i.e. no sub-folders.
- *ZIP file example:*
  - *B1-00001.wav*
  - *B1-00002.wav*
  - *B1-00003.wav*
  - *B1-00004.wav*
  - *labels.txt*

**Processed WAV files**

- this Notebook allows you to download and save your processed WAV files to Google Drive. Processed WAV files that are directly prepared for training have these **attributes**:
  - they are **MONO** files with a sample rate of **24k**
  - they have **leading and trailing silences trimmed**
- furthermore, the ZIP file with processed WAV files also contains the following:
  - **metadata.csv** file that's compatible with LJSpeech CSV format. This file is not directly used but is kept as a reference, so you can see in plain text what sentences is the TTS being trained on.
  - **metadata_phonetical.csv** file that is a direct phonetical representation of the metadata.csv file. This file is not directly used for training but training and validation data are devised from it.
  - **training.csv** file which contains training data for StyleTTS2
  - **validation.csv** file which contains validation data for StyleTTS2
- structure of this ZIP file is different than the raw WAV files ZIP file
- *ZIP file example:*
  - *metadata.csv*
  - *metadata_phonetical.csv*
  - *training.csv*
  - *validation.csv*
  - *wavs/B1-00001.wav*
  - *wavs/B1-00002.wav*
  - *wavs/B1-00003.wav*
  - *wavs/B1-00004.wav*

In [ ]:
#################
# Settings Form #
#################

restoration_zip_path = "" # @param {type:"string"}

wav_files_path = "style_tts.zip" # @param {type:"string"}

# @markdown ***WAV files prefix:***
# @markdown <br><small>Prefix for each of the WAV files, as described in [Section 0](#scrollTo=6DEC0-d48M-G).</small>
wavs_prefix = 'wav1-' # @param {type:"string"}

# @markdown ***WAV files in ZIP archive are already prepared:***
# @markdown <br><small>If you're resuming training or have previously prepared your WAV files, so they conform to the structure generated by this Notebook (see [Section 0](#scrollTo=6DEC0-d48M-G)), tick this checkbox. In such case, this Notebook will not try to convert your WAV files, nor will trim leading and trailing silences from them.</small>
wav_files_prepared_for_training = False # @param {type:"boolean"}

# @markdown ***Silence trimming treshold:***
# @markdown <br><small>This value is used as the "*-af silenceremove*" ffmpeg parameter.</small>
# @markdown <br><small>I found the default (-60dB) to be good enough to trim silences from well-recorded, denoised WAV files.</small>
# @markdown <br><small>If this trims too much of the audio, try -96dB for a more lenient trim.</small>
db_trim_value = "-60dB" # @param {type:"string"}

# @markdown ***Phonemization language:***
# @markdown <br><small>This language will be used to convert your text into phonemes.</small>
# @markdown <br><small>This value must be one of the languages compatible with the phonemizer: https://pypi.org/project/phonemizer/</small>
# @markdown <br><small>For example, if you're using the default espeak phonemization: https://github.com/espeak-ng/espeak-ng/blob/master/docs/languages.md</small>
phonemization_language = 'en-us' # @param {type:"string"}

# @markdown ***Training VS validation data percentage:***
# @markdown <br><small>This value says how much % of the total data will be used for training.</small>
# @markdown <br><small>Default value: 90 (i.e. 90% training data, 10% validation data)</small>
training_data_percentage = 90 # @param {type:"integer"}

# @markdown ***Labels text replacements:***
# @markdown <br><small>In case some words in your labels.txt file are pronounced differently than they appear, set your replacement rules here.</small>
# @markdown <br><small>This text is in a JSON form and needs to be updated in the same form in order for this cell to keep working.</small>
# @markdown <br><small>The example default are some replacements from a fantasy book, so you can see the syntax of the JSON value.</small>
text_replacements = { "Bannick":"ban-nick", "Banshee": "ban-shee", "Cait Sidhe": "kay-th shee", "Coblynau": "cob-lee-now", "Daoine": "doon-ya", "Djinn": "jin", "Glastig": "glass-tig", "Gwragen": "guh-war-a-gen", "Kelpie": "kel-pee", "Kitsune": "kit-soon", "Lamia": "lay-me-a", "Manticore": "man-tee-core", "Nixie": "nix-ee", "Peri": "pear-ee", "Piskie": "piss-key", "Pixie": "pix-ee", "Puca": "puh-ca", "Roane": "ro-an", "Selkie": "sell-key", "Sidhe": "shee", "Silene": "sigh-lean", "The Luidaeg": "the lou-sha-k", "Tuatha": "tootha", "de Dannan": "day danan", "Tybalt": "tiebolt", "Tylwyth": "till-with", "Teg": "teeg", "Undine": "un-deen", "Will o' Wisp": "will-oh wisp", "de Dannan": "day danan" } # @param {type:"raw"}


# 2. Required programs and libraries

This section will install everything you'll need later.

In [ ]:
# install system packages (Debian, Ubuntu)
!sudo apt -y install curl ffmpeg sox libtool musl-dev gcc-multilib g++-multilib espeak espeak-ng sox zip unzip tensorrt
!sudo apt update && sudo apt -y upgrade

# install Python dependencies
!pip install pyloudnorm soundfile nltk phonemizer wget numpy tqdm matplotlib ipython pandas accelerate tensorrt

!pip install -r StyleTTS2/requirements.txt

print('done')

# 3. Labels and WAV Files Preparation

In [ ]:
remove_downloaded_wav_file = False

if wav_files_path == "":
  raise Exception("The wav_files_path setting is empty. Please set this option in section 1, then run the cell in section 1 and re-run this cell again.")

cp $wav_files_path ./wavs.zip

# extract WAV files from the ZIP file into a local folder
mkdir wavs
unzip style_tts.zip -d wavs
if remove_downloaded_wav_file == True:
  rm -f wavs.zip

# prepare WAV files from original ones to a safely-renamed ones
# and phonemize the labels.txt file, so it can be used with the StyleTTS2 system
import os
import nltk
import phonemizer
import pandas as pd
import csv
from nltk.tokenize import word_tokenize
from shutil import copyfile, rmtree
from pathlib import Path

if wav_files_prepared_for_training == False:
  nltk.download('punkt')
  nltk.download('punkt_tab')
  global_phonemizer = phonemizer.backend.EspeakBackend( language=phonemization_language, preserve_punctuation=True, with_stress=True )

def text_to_phonemes(text):
  text = text.strip()
  #print("Text before phonemization: ", text)
  ps = global_phonemizer.phonemize([text])
  #print("Text after phonemization: ", ps)
  ps = word_tokenize(ps[0])
  ps = ' '.join(ps)
  #print("Final text after tokenization: ", ps)
  return ps

wavs_counter = 1
wavs_export_path = './wavs'
wavs_processed_path = './wavs_processed'
wavs_final_path = './wavs_final'
wavs_filtered_path = './wavs_filtered'
input_filename = './wavs/labels-large-fixed.txt'
output_filename_clean = './wavs_final/metadata.csv'
output_filename_phonetical = './wavs_final/metadata_phonetical.csv'

# delete and recreate wavs_final_path when rerunning
if wav_files_prepared_for_training == False:
  if os.path.exists(wavs_final_path):
    rmtree(wavs_final_path)
  os.mkdir(wavs_final_path)

  # delete and recreate wavs_processed_path when rerunning
  if os.path.exists(wavs_processed_path):
    rmtree(wavs_processed_path)
  os.mkdir(wavs_processed_path)

  fp_clean = open(output_filename_clean, 'w')
  fp_phonetical = open(output_filename_phonetical, 'w')

  df = pd.read_csv(input_filename, sep='\t', encoding='utf-8', usecols=[0, 1, 2], names=['Start', 'End', 'Description'], quoting=csv.QUOTE_NONE)

  for sentence in df['Description'].to_list():
    if not pd.isnull(sentence):
      # make sure this sentence does not contain new line, as that would suggest and opening and closing quotes on different lines
      # and thus read_csv() treating this as a single line, while it's actually multiple
      for sub_sentence in sentence.splitlines():
          # replace special words
          for original_text, replacement_text in text_replacements.items():
            sub_sentence = sub_sentence.replace( original_text, replacement_text )

          # adjust file name
          real_counter = str( wavs_counter )
          if wavs_counter < 10:
            real_counter = '0000' + real_counter
          elif wavs_counter < 100:
            real_counter = '000' + real_counter
          elif wavs_counter < 1000:
            real_counter = '00' + real_counter
          elif wavs_counter < 10000:
            real_counter = '0' + real_counter

          wav_path_orig = wavs_prefix + real_counter + ".wav"
          wav_path = wav_path_orig.replace(" ", "_").replace("M", "m")
          wav_filename = wav_path.replace(".wav", "")
          copyfile(wavs_export_path + '/' + wav_path_orig, wavs_processed_path + '/' + wav_path)
          print(wavs_prefix + real_counter + '.wav (' + ( wavs_processed_path + '/' + wav_path ) + '): ' + sentence)

          # write the sentence into the original LJSpeech metadata form file
          fp_clean.write(f"{wav_filename}|{sub_sentence}|{sub_sentence}\n")

          # transliterate using espeak's phonemizer to conform with StyleTTS2's expected input
          phonemized = text_to_phonemes( sub_sentence )
          fp_phonetical.write(f"{wav_filename}.wav|{phonemized}|0\n")
          wavs_counter = wavs_counter + 1
  fp_clean.close()
  fp_phonetical.close()
else:
  # we've uploaded prepared WAV files, just rename the folder
  mv ./wavs ./wavs_final


# process WAV files, trimming leading and ending silences,
# converting their sample rate to 24k and changing them to mono
if wav_files_prepared_for_training == False:
  from multiprocessing import Pool, cpu_count, Manager
  import random

  # Function to process a single file
  def process_file(args):
    file, processing_index, lock, total_files = args

    with lock:
      processing_index.value += 1
      print(f"{processing_index.value} / {total_files}: processing {file}, output into {file.replace(wavs_processed_path.replace('./', ''), wavs_final_path)}")

    rnd = random.randint(0, 100000)
    temp1 = f"{wavs_processed_path}/temp1_{rnd}.wav"
    temp2 = f"{wavs_processed_path}/temp2_{rnd}.wav"
    temp3 = f"{wavs_processed_path}/temp3_{rnd}.wav"

    os.system(f"ffmpeg -i {file} -af silenceremove=1:0:{db_trim_value} {temp1} > /dev/null 2>&1")
    os.system(f"ffmpeg -i {temp1} -af areverse {temp2} > /dev/null 2>&1")
    os.system(f"ffmpeg -i {temp2} -af silenceremove=1:0:{db_trim_value} {temp3} > /dev/null 2>&1")
    os.system(f"ffmpeg -y -i {temp3} -map_metadata -1 -af areverse -ar 24000 -ac 1 " + file.replace( wavs_processed_path.replace( "./", "" ), wavs_final_path ) + " > /dev/null 2>&1")

    os.remove(temp1)
    os.remove(temp2)
    os.remove(temp3)
    print(f"finished processing {file}\n")

  paths = Path( wavs_processed_path ).glob("**/*.wav")
  file_list = [ str(file) for file in paths if file.suffix == '.wav' ]

  manager = Manager()

  processing_index = manager.Value('i', 0)
  lock = manager.Lock()
  total_files = len(file_list)

  # Create list of arguments for each process
  args_list = [(file, processing_index, lock, total_files) for file in file_list]

  num_processes = cpu_count()
  print(f"\nusing {num_processes} processes to update process WAV files\n")
  with Pool(num_processes) as pool:
    pool.map(process_file, args_list)


  # randomize metadata file, so the phrases from labels.txt are fed to this TTS system in random sequence
  shuf ./wavs_final/metadata_phonetical.csv > ./wavs_final/shuffled.csv


  # prepare a script that would split the resulting phonetical transcription
  # into training and validation file by a ratio of 90:10 by default (10% validation data, 90% training data)
  # (this can be changed by changing the training_data_percentage variable on top of this Notebook)
  with open('splitter.sh', 'w') as file:
    file.write('''# Specify the names for the training and validation files
training_file="training.csv"
validation_file="validation.csv"

# Calculate the line count for the split
total_lines=$(wc -l ./wavs_final/shuffled.csv | cut -d" " -f1)
training_lines=$((total_lines * {training_data_percentage} / 100))

# Use the split command with the specified names for output files
cd ./wavs_final && split -l $training_lines shuffled.csv "$training_file"_

# Rename the generated files to the desired names
mv ./"$training_file"_aa ./"$training_file"
mv ./"$training_file"_ab ./"$validation_file"
rm -f ./shuffled.csv'''.format(training_data_percentage = training_data_percentage));

  sudo chmod +x ./splitter.sh
  ./splitter.sh


  # cleanup temporary folders and move WAV files into a TTS structure
  rm -rf ./wavs
  rm -rf ./wavs_processed
  rm -rf ./wavs_tmp
  mkdir ./wavs_final/wavs
  !mv ./wavs_final/*.wav ./wavs_final/wavs

print('done')

# 4. WAV Files Validation (optional, skip when restoring)

In this section, you can validate the quality of your WAV files and see their durations. You can optionally choose to create a subset of all WAV files that conform to your minimum and maximum audio length preference (see the last step of this section).

In [ ]:
# download and compile the Waveform Amplitude Distribution Analysis tool (WADA)
# used to analyze the Signal-to-Noise Ratio Estimation in your WAV files
# ... nothing here is configurable, just run this cell and check the output if you like
!wget http://www.cs.cmu.edu/~robust/archive/algorithms/WADA_SNR_IS_2008/WadaSNR.tar.gz
!mkdir WADA_SNR_IS_2008/ && tar -xvf WadaSNR.tar.gz -C WADA_SNR_IS_2008
!cd ./WADA_SNR_IS_2008/Build/ && rm -rf *.o && make clean && make

import sys
import glob
import subprocess
import tempfile
import IPython
import soundfile as sf
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from matplotlib import pylab as plt
%matplotlib inline

import multiprocessing
cores = multiprocessing.cpu_count() # Count the number of cores in a computer
cores

# Set the meta parameters
DATA_PATH = wavs_final_path + "/wavs"
NUM_PROC = cores
CURRENT_PATH = "./"

# define the SNR compute function
def compute_file_snr(file_path):
    """ Convert given file to required format with FFMPEG and process with WADA."""
    _, sr = sf.read(file_path)
    new_file = file_path.replace(".wav", "_tmp.wav")
    print(new_file)
    if sr != 16000:
      #command = f'ffmpeg -i "{file_path}" -ac 1 -acodec pcm_s16le -y -ar 16000 "{new_file}"'
      command = f'ffmpeg -i "{file_path}" -ac 1 -acodec pcm_s16le -y -ar 16000 "{new_file}"'
    else:
      command = f'cp "{file_path}" "{new_file}"'
    os.system( command + " > /dev/null 2>&1")
    #os.system(command)
    #subprocess.run( command, shell=True )
    output = !./WADA_SNR_IS_2008/Exe/WADASNR -i {new_file} -t ./WADA_SNR_IS_2008/Exe/Alpha0.400000.txt -ifmt mswav
    snr = float(output[-2].split()[-2])
    os.system(f'rm "{new_file}"  > /dev/null 2>&1')
    return snr, file_path

# define SNR results output function
def output_snr_with_audio(idx):
    file_idx = file_idxs[idx]
    file_name = file_names[file_idx]
    wav, sr = sf.read(file_name)
    # multi channel to single channel
    if len(wav.shape) == 2:
        wav = wav[:, 0]
    print(f" > {file_name} - snr:{snrs[file_idx]}")
    IPython.display.display(IPython.display.Audio(wav, rate=sr))

# run WADA script
!./WADA_SNR_IS_2008/Exe/WADASNR -i ./WADA_SNR_IS_2008/SampleCorrupt/sb01_00dB_White.sph -t ./WADA_SNR_IS_2008/Exe/Alpha0.400000.txt -ifmt nist

# validate number of WAV files
wav_files = glob.glob(f"{DATA_PATH}/*.wav", recursive=True)
print(f"Number of wav files found {len(wav_files)}")
print('done')

In [ ]:
# nothing to configure here either - just run and check the output if you want to
if NUM_PROC == 1:
    file_snrs = [None] * len(wav_files)
    for idx, wav_file in tqdm(enumerate(wav_files)):
        tup = compute_file_snr(wav_file)
        file_snrs[idx] = tup
else:
    with Pool(NUM_PROC) as pool:
        file_snrs = list(tqdm(pool.imap(compute_file_snr, wav_files), total=len(wav_files)))

snrs = [tup[0] for tup in file_snrs]

error_idxs = np.where(np.isnan(snrs) == True)[0]
error_files = [wav_files[idx] for idx in error_idxs]

file_snrs = [i for j, i in enumerate(file_snrs) if j not in error_idxs]
file_names = [tup[1] for tup in file_snrs]
snrs = [tup[0] for tup in file_snrs]
file_idxs = np.argsort(snrs)

print("")
print("")
print(f"Average SNR of the dataset:{np.mean(snrs)}")

In [ ]:
# find worse SNR files
N = 3  # number of files to fetch
for i in range(N):
  #file_idx = file_idxs[i]
  #print(file_names[file_idx] + '|' + str(snrs[file_idx]))
  output_snr_with_audio(i)

In [ ]:
# find best recordings
N = 3  # number of files to fetch
for i in range(N):
    output_snr_with_audio(-i-1)

In [ ]:
# visualize SNRS
plt.hist(snrs, bins=100)

In [ ]:
# save durations of your WAV files to allow for filtering out those that are too short or too long below
#!echo "filename|duration" > wavdurations.csv;
!echo "" > wavdurations.csv;

!for file in ./wavs_final/wavs/*.wav; do duration=$(eval soxi -D "$file"); echo "${file}|$duration" >> wavdurations.csv; done

#!cat wavdurations.csv

# @markdown ***WAV files duration check and filtering - skip when restoring***
# @markdown <br><small>WAV files should generally be longer than 1 second and shorter than about 10 - 12 seconds.</small>
# @markdown <br><small>You can set your own limits and run this cell to see whether any of your WAV files are either too short or too long.</small>
min_wav_duration_seconds = 1 # @param {type:"number"}
max_wav_duration_seconds = 14 # @param {type:"number"}

# @markdown <br><small>If you'd like to save only the files that conform to the min-max interval set above (along with updated labels.txt file) into a separate folder, tick this checkbox.</small>
save_valid_files_separately = True # @param {type:"boolean"}

files_too_short = {}
files_too_long = {}
min_seconds = 0
max_seconds = 0
some_files_filtered_out = False

wavs_filtered_path = './wavs_filtered'

if save_valid_files_separately == True:
  !rm -rf $wavs_filtered_path
  !mkdir $wavs_filtered_path && cd $wavs_filtered_path && mkdir wavs
  !cp $wavs_final_path/training.csv $wavs_filtered_path
  !cp $wavs_final_path/validation.csv $wavs_filtered_path

  # load all lines of training and validation files,
  # so we can remove the ones we don't want to see in them
  with open( wavs_filtered_path + '/training.csv', 'r', newline = '', encoding='UTF8') as f:
    lines_training = f.readlines()

  with open( wavs_filtered_path + '/validation.csv', 'r', newline = '', encoding='UTF8') as f:
    lines_validation = f.readlines()

df = pd.read_csv('wavdurations.csv', sep='|', encoding='utf-8', usecols=[0, 1], names=['Filename', 'Duration'])
for row in zip(df['Filename'].to_list(), df['Duration'].to_list()):
  should_copy_this_file = True

  if ( float( row[ 1 ] ) < min_wav_duration_seconds ):
    files_too_short[ row[ 0 ] ] = row[ 1 ]
    should_copy_this_file = False

  if ( float( row[ 1 ] ) > max_wav_duration_seconds ):
    files_too_long[ row[ 0 ] ] = row[ 1 ]
    should_copy_this_file = False

  if ( float( row[ 1 ] ) < min_seconds or min_seconds == 0 ):
    min_seconds = float( row[ 1 ] )

  if ( float( row[ 1 ] ) > max_seconds ):
    max_seconds = float( row[ 1 ] )

  filename_origin = row[ 0 ]

  if save_valid_files_separately == True:
    if should_copy_this_file:
      # copy this file into the filtered files folder
      filename_target = filename_origin.replace( wavs_final_path, wavs_filtered_path )
      !cp $filename_origin $filename_target
    else:
      some_files_filtered_out = True
      # remove this file from training and validation arrays
      tmp_training = [];
      tmp_validation = []
      filename_clear = filename_origin.replace( wavs_final_path + '/wavs/', '' )
      print( "filtering out " + filename_clear )

      for line in lines_training:
        if line.split('|')[0] != filename_clear:
          tmp_training.append( line )

      lines_training = tmp_training

      for line in lines_validation:
        if line.split('|')[0] != filename_clear:
          tmp_validation.append( line )

      lines_validation = tmp_validation

# rewrite training and validation files, if we removed files from them
if some_files_filtered_out == True:
  with open( wavs_filtered_path + '/training.csv', 'w', newline = '', encoding='UTF8') as f:
    for line in lines_training:
      f.write( line )

  with open( wavs_filtered_path + '/validation.csv', 'w', newline = '', encoding='UTF8') as f:
    for line in lines_validation:
      f.write( line )

  print( "WAVs length filtering successful, relevant files stored under " + wavs_filtered_path )

print( "\n" )
print("Minimum seconds in a WAV file found: " + str( min_seconds ) )
print("Maximum seconds in a WAV file found: " + str( max_seconds ) )
print( "\n" )

if ( len( files_too_short ) ):
  print( "WAV files too short:" )
  for file_path, file_duration in files_too_short.items():
    print( " - " + file_path + " = " + str( file_duration ) + "s" )

print( "\n" )

if ( len( files_too_long ) ):
  print( "WAV files too long:" )
  for file_path, file_duration in files_too_long.items():
    print( " - " + file_path + " = " + str( file_duration ) + "s" )

print('done')

# 5. WAV Files Backup (optional, skip when restoring)

Here you can backup your processed WAV files and copy them onto Google Drive (if connected) or download a backup copy of the ZIP file.

In [ ]:
# @markdown ***Path for the WAVS backup ZIP file:***
# @markdown <br><small>This is the relative path from your Drive root (e.g. *tts/wavs_processed.zip*)</small>
# @markdown <br><small>Leave empty if you only want to zip-up your WAV files and download them manually.</small>
wav_files_backup_path = "tts/wavs_backup.zip" # @param {type:"string"}
make_backup_from = "Filtered WAV files (result of running length check in Step 4)" # @param ["Unfiltered processed WAV files (Step 4 length check not ran)", "Filtered WAV files (result of running length check in Step 4)"]

# local WAV files backup file name
wavs_backup_local_file_name = "wavs_backup.zip"

# remove old file if we're backing up again
!rm -f ./$wavs_backup_local_file_name

# make backup of the resulting WAV files
if make_backup_from == "Unfiltered processed WAV files (Step 4 length check not ran)":
  !cd $wavs_final_path && zip -r ../$wavs_backup_local_file_name *
else:
  !cd $wavs_filtered_path && zip -r ../$wavs_backup_local_file_name *

print( "You can download the file " + wavs_backup_local_file_name + " directly from the root folder of this Notebook." )


# 6. TTS Preparation (skip when restoring)

In [ ]:
!pip install huggingface_hub
!pip install pickleshare
!git clone https://github.com/yl4579/StyleTTS2.git
%cd StyleTTS2

In [ ]:
from huggingface_hub import hf_hub_download, HfApi, Repository
import os

# Define the model repository and file paths
model_repo = "yl4579/StyleTTS2-LibriTTS"
model_path = "Models/LibriTTS"
local_dir = "."
target_files = ["config.yml","epochs_2nd_00020.pth"]

# Create a local dir to store the downloaded files 
os.makedirs(local_dir, exist_ok=True)

# Download the files from the Huggingface Model Repository
downloaded_files = []
for file_name in target_files:
    file_path = hf_hub_download(repo_id=model_repo, filename=f"{model_path}/{file_name}", local_dir=local_dir)
    downloaded_files.append(file_path)

print(f"Downloaded Files:{downloaded_files}")

In [ ]:
!ls
import os

In [ ]:
!sudo mv /wavs_filtered/* StyleTTS2/Data

#remove the original StyleTTS2 training and validation files, as we have our own
!rm -f /StyleTTS2/Data/train_list.txt
!rm -f /StyleTTS2/Data/val_list.txt

#copy our own training and validation files into the StyleTTS2 Data folder
!sudo mv StyleTTS2/Data/training.csv StyleTTS2/Data/train_list.txt
!sudo mv StyleTTS2/Data/validation.csv StyleTTS2/Data/val_list.txt

print('Done...!')

In [ ]:
if os.getcwd().split('/')[-1] != "StyleTTS2":
    %cd StyleTTS2

# Updating the config values in the config_ft.yml file
config_path = "Configs/config_ft.yml"

import yaml
config = yaml.safe_load(open(config_path))

In [ ]:
config['data_params']['root_path'] = "Data/wavs"
config['data_params']['train_data'] = "Data/train_list.txt"
config['data_params']['val_data'] = "Data/val_list.txt"
config['epochs'] = 100
config['batch_size'] = 3
config['max_len'] = 500
config['loss_params']['diff_epoch'] = 20
config['loss_params']['joint_epoch']= 30
config["loss_params"]["TMA_epoch"] = 10
config['save_freq'] = 50
config["load_only_params"] = True
config['log_dir'] = "Models/LibriTTS-fft"
config['slmadv_params']['batch_percentage'] = 2 
# # update config values
# !sed -i 's/batch_size: 16/batch_size: {batch_size}/g' ./StyleTTS2/Configs/config.yml
# !sed -i 's/epochs_1st: 200/epochs_1st: {epochs_stage_1}/g' ./StyleTTS2/Configs/config.yml
# !sed -i 's/epochs_2nd: 100/epochs_2nd: {epochs_stage_2}/g' ./StyleTTS2/Configs/config.yml
# !sed -i 's/save_freq: 2/save_freq: {save_after}/g' ./StyleTTS2/Configs/config.yml
#!sed -i 's/root_path: "\/local\/LJSpeech-1.1\/wavs"/root_path: "Data\/wavs"/g' ./StyleTTS2/Configs/config.yml

# # optionally, change max_len if you're running out of GPU RAM
# # ... don't go below 200 or the quality will really suffer
# # ... a value of 800 is optimal and would cover most training material up to 10-12 seconds per WAV file
# !sed -i 's/max_len: 400/max_len: {max_len}/g' ./StyleTTS2/Configs/config.yml

with open(config_path, "w") as outfile:
    yaml.dump(config, outfile, default_flow_style=False)
    
print('Done!')

In [ ]:
!sudo apt-get install -y tensorrt && sudo apt -y upgrade
!pip install tensorrt accelerate

# 7. TTS Training STAGE 1

In [ ]:
!sudo apt-get install -y python3-dev
!pip install -r requirements.txt --no-cache-dir --upgrade

In [ ]:
!pip install matplotlib-inline --no-cache-dir --upgrade
!pip install tensorboard

In [ ]:
!sudo apt-get install libsndfile1
!pip install --upgrade soundfile

# Being fine-tuning using accelerate

In [ ]:
# begin STAGE 1 training
!accelerate launch --mixed_precision=fp16 --num_processes=1 train_finetune_accelerate.py --config_path ./Configs/config_ft.yml

# Setting up Inferencing the fine-tuned model

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
import random
random.seed(0)
import numpy as np
np.random.seed(0)

# load packages
import time
import random
import yaml
from munch import Munch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import torchaudio
import librosa
from nltk.tokenize import word_tokenize

from models import *
from utils import *
from text_utils import TextCleaner
textclenaer = TextCleaner()

%matplotlib inline

to_mel = torchaudio.transforms.MelSpectrogram(
    n_mels=80, n_fft=2048, win_length=1200, hop_length=300)
mean, std = -4, 4

def length_to_mask(lengths):
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask+1, lengths.unsqueeze(1))
    return mask

def preprocess(wave):
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor = to_mel(wave_tensor)
    mel_tensor = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor

def compute_style(path):
    wave, sr = librosa.load(path, sr=24000)
    audio, index = librosa.effects.trim(wave, top_db=30)
    if sr != 24000:
        audio = librosa.resample(audio, sr, 24000)
    mel_tensor = preprocess(audio).to(device)

    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))

    return torch.cat([ref_s, ref_p], dim=1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load phonemizer
import phonemizer
global_phonemizer = phonemizer.backend.EspeakBackend(language='en-us', preserve_punctuation=True,  with_stress=True)

In [ ]:
model_folder_path = "Models/LibriTTS-fft"
config = yaml.safe_load(open(model_folder_path+"/config_ft.yml"))

# load pretrained ASR model
ASR_config = config.get('ASR_config', False)
ASR_path = config.get('ASR_path', False)
text_aligner = load_ASR_models(ASR_path, ASR_config)

In [ ]:
# load pretrained F0 model
F0_path = config.get('F0_path', False)
pitch_extractor = load_F0_models(F0_path)

# load BERT model
from Utils.PLBERT.util import load_plbert
BERT_path = config.get('PLBERT_dir', False)
plbert = load_plbert(BERT_path)

model_params = recursive_munch(config['model_params'])
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]

In [ ]:
files = [f for f in os.listdir(model_folder_path) if f.endswith('.pth')]
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

In [ ]:
print(sorted_files)

In [ ]:
params_whole = torch.load(model_folder_path + "/" + sorted_files[-1], map_location='cpu', weights_only=True)

if 'net' in params_whole:
    params = params_whole['net']
else:
    params = params_whole

In [ ]:
for key in model:
    if key in params:
        print('%s loaded' % key)
        try:
            model[key].load_state_dict(params[key])
        except:
            from collections import OrderedDict
            state_dict = params[key]
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] # remove `module.`
                new_state_dict[name] = v
            # load params
            model[key].load_state_dict(new_state_dict, strict=False)
#             except:
#                 _load(params[key], model[key])

In [ ]:
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

In [ ]:
sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0), # empirical parameters
    clamp=False
)

In [ ]:
def inference(text, ref_s, alpha = 0.3, beta = 0.7, diffusion_steps=10, embedding_scale=1):
    text = text.strip()
    ps = global_phonemizer.phonemize([text])
    ps = word_tokenize(ps[0])
    ps = ' '.join(ps)
    tokens = textclenaer(ps)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)

    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask = length_to_mask(input_lengths).to(device)

        t_en = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
                noise = torch.randn((1, 256)).unsqueeze(1).to(device),
                embedding=bert_dur,
                embedding_scale=embedding_scale,
                features=ref_s, # reference from the same speaker as the embedding
                num_steps=diffusion_steps
            ).squeeze(1)

        s = s_pred[:, 128:]
        ref = s_pred[:, :128]

        ref = alpha * ref + (1 - alpha)  * ref_s[:, :128]
        s = beta * s + (1 - beta)  * ref_s[:, 128:]

        d = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)

        x, _ = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)

        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)

        pred_aln_trg = torch.zeros(input_lengths, int(pred_dur.sum().data))
        c_frame = 0
        for i in range(pred_aln_trg.size(0)):
            pred_aln_trg[i, c_frame:c_frame + int(pred_dur[i].data)] = 1
            c_frame += int(pred_dur[i].data)

        # encode prosody
        en = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(en)
            asr_new[:, :, 0] = en[:, :, 0]
            asr_new[:, :, 1:] = en[:, :, 0:-1]
            en = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)

        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == "hifigan":
            asr_new = torch.zeros_like(asr)
            asr_new[:, :, 0] = asr[:, :, 0]
            asr_new[:, :, 1:] = asr[:, :, 0:-1]
            asr = asr_new

        out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))


    return out.squeeze().cpu().numpy()[..., :-50]

## Finally, Syntesize Speech!!

In [ ]:
text = '''Replace this with the passage you want to synthesize.'''

In [ ]:
# get a random reference in the training set, note that it doesn't matter which one you use
path = "Data/wavs/wav1-00061.wav"

# this style vector ref_s can be saved as a parameter together with the model weights
ref_s = compute_style(path)

In [ ]:
start = time.time()
wav = inference(text, ref_s, alpha=0.3, beta=0.3, diffusion_steps=10, embedding_scale=1)
rtf = (time.time() - start) / (len(wav) / 24000)
print(f"RTF = {rtf:5f}")

import IPython.display as ipd
display(ipd.Audio(wav, rate=24000, normalize=True))

# 8. TTS Backup (optional but recommended)

In [ ]:
%cd ..

In [ ]:
# make backup of the progress
!zip -r -2 model_backup_stage_1.zip StyleTTS*